# 캡스톤 실습 뼈대

- 구분: 수강생 실습본
- 원본: [`starter.py`](starter.py)
- 연결 교안: [4-3-research-automation-multi-agent-system.pdf](../../../../docs/raw/Chapter02/day02/4-3-research-automation-multi-agent-system.pdf)

원본 Python 코드는 수정 없이 아래의 한 코드 셀에 담았습니다. 설명과 실행 준비는 별도 셀입니다. **모두 실행** 대신 필요한 셀을 직접 선택하세요.


## 학습 목표

1. Supervisor·Researcher·Writer의 책임과 HITL 승인 위치를 설계한다.
2. 코드를 실행하기 전 입력, 출력, 외부 호출과 파일 변경 가능성을 설명한다.
3. CodeMate가 만든 결과를 원본과 교안에 비춰 검토한다.


## 코드를 읽는 순서

도구 → State → 평가 스키마 → 에이전트 → 그래프 → main 순서로 TODO를 찾는다.

교안은 개념과 목표를 설명합니다. 코드에서 실제 구현된 부분과 TODO를 구분하세요.


## 실행 준비

저장소 안에서 VS Code로 열고 프로젝트 `.venv` 커널을 선택합니다. 의존성은 `uv sync`로 준비합니다. 사내망에서는 사내 LLM 환경을, 사외에서는 허용된 provider를 준비합니다. 자격증명 값은 셀에 적지 않습니다. 아래 준비 셀은 저장소 루트를 찾고 `__file__`을 원본 Python 파일 경로로 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys
_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents) if (p / "pyproject.toml").is_file() and (p / "config" / "llm.py").is_file()), None)
if _repo is None:
    raise FileNotFoundError("저장소 안에서 노트북을 열어 주세요.")
os.chdir(_repo)
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
__file__ = str(_repo / 'src/raw/Chapter02/06-capstone/starter.py')
import __main__
sys.modules.setdefault("notebook_source", __main__)
__name__ = "notebook_source"  # 스크립트 전용 실행 부분은 아래에서 따로 실행
print(f"실행 기준 폴더: {_repo}")


## 원본 코드 — 직접 실행

**조건과 부작용:** TODO가 남은 미완성 파일이다. 배포본에는 `mcp_server.py`와 `skills/`가 없고 정답본에만 있다. README는 Streamlit UI에 15점을 배점하지만 정답본에도 구현이 없다. 실행 시 API·파일 작업이 생길 수 있다.

준비 셀을 먼저 실행하고 필요한 서비스·설정을 확인하세요. 다음 셀은 `manual-run`으로 표시했습니다. 원본 전체가 한 번에 실행되므로 API 요청, 파일 변경 또는 대기가 생길 수 있습니다.


In [ ]:
"""
캡스톤: 리서치 자동화 멀티에이전트 시스템

4일 과정 통합 실습:
  LangGraph(멀티에이전트·HITL) + create_deep_agent(Harness) + MCP + RAGaaS + Langfuse

흐름:
  사용자 입력 → Supervisor → Researcher(MCP+RAG+웹검색) → [HITL 승인]
               → Writer(SDS 보고서 작성) → 최종 출력

실습 파일: src/raw/Chapter02/06-capstone/starter.py
정답 참고: src/solution/Chapter02/06-capstone/solution.py
정답용 MCP 서버: src/solution/Chapter02/06-capstone/mcp_server.py

starter.py는 구현할 부분이 남아 있는 실습 뼈대다.
사외에서는 RAG_PROVIDER=mock으로 RAGaaS 응답 형식을 모의 실행한다.
"""

import asyncio
import json
import os
from pathlib import Path
from typing import Annotated, Literal

import requests
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from pydantic import BaseModel
from typing_extensions import TypedDict

from config.llm import get_llm
from config.ragaas import post_ragaas

load_dotenv()

CAPSTONE_DIR = Path(__file__).parent

# ── 도구 정의 ──────────────────────────────────────────────────────────────────
# TODO: 사용할 도구를 정의

@tool
def search_web(query: str) -> str:
    """웹에서 최신 정보를 검색한다."""


@tool
def rag_search(query: str) -> str:
    """내부 지식베이스에서 관련 문서를 검색한다. 삼성 사내망에서는 RAGaaS API를 사용한다."""
    # TODO: post_ragaas('retrieve-rrf', payload)를 Coding AI에게 구현하게 하고
    # RAG_PROVIDER=mock일 때 결과에 [MOCK RAGaaS]가 표시되는지 검토하세요.


# ── State ─────────────────────────────────────────────────────────────────────

# TODO: 필요한 필드를 추가하세요.
class State(TypedDict):
    messages: Annotated[list, add_messages]




# ── 평가 스키마 ───────────────────────────────────────────────────────────────

class EvalDimension(BaseModel):
    score: float  # 0.0 ~ 1.0 (0.1 단위로 엄격 채점)
    reason: str


class ReportEvaluation(BaseModel):
    relevance: EvalDimension     # 주제 관련성
    faithfulness: EvalDimension  # 리서치 근거 충실도
    completeness: EvalDimension  # 보고서 구성 완성도
    clarity: EvalDimension       # 문장·구조 명확성


def evaluate_report(
    topic: str, research: str, report: str, langfuse_client
) -> ReportEvaluation:
    """LLM Judge: 4개 기준으로 보고서를 평가하고 Langfuse에 지표별 점수를 기록한다.

    1.0은 완벽할 때만 — 0.1 단위 엄격 채점으로 의미 있는 점수 분포를 만든다.
    """
    langfuse = get_client()
    langfuse_handler = CallbackHandler()
    langfuse_config = {'callbacks': [langfuse_handler]}

    eval_llm = get_llm().with_structured_output(ReportEvaluation)
    # CallbackHandler가 LLM 호출을 자동으로 트레이스로 기록한다
    eval_handler = CallbackHandler()

    # TODO: 평가를 진행하고 Langfuse에 점수를 기록

# ── Researcher 에이전트 ────────────────────────────────────────────────────

researcher_backend = FilesystemBackend(
    root_dir=str(CAPSTONE_DIR), virtual_mode=True,
)
# TODO: 자료조사 skill을 가진 Researcher Agent


# ── Writer 에이전트 ────────────────────────────────────────────────────────
writer_backend = FilesystemBackend(
    root_dir=str(CAPSTONE_DIR), virtual_mode=True,
)
# TODO: 보고서 작성 skill을 가진 Writer Agent


# ── LangGraph 노드 정의 ────────────────────────────────────────────────────
class RouteDecision(BaseModel):
    next: Literal['researcher', 'writer', 'END']
    reason: str


# TODO: Supervisor 노드 정의
supervisor_llm = get_llm().with_structured_output(RouteDecision)


def supervisor_node(state: State) -> dict:
    pass


# TODO: Researcher 노드 정의
async def researcher_node(state: State) -> dict:
    pass


# TODO: Writer 노드 정의
async def writer_node(state: State) -> dict:
    pass


def route(state: State) -> str:
    return state.get('next', 'END')


builder = StateGraph(State)
builder.add_node('supervisor', supervisor_node)
builder.add_node('researcher', researcher_node)
builder.add_node('writer', writer_node)
# TODO: 그래프 조립 

app = builder.compile()


# ── 메인 함수 (async: MCP 클라이언트가 비동기) ──────────────────────────────────

async def main(topic: str):
    

    config = {'configurable': {'thread_id': 'capstone-session'}}
    initial_state = {
        'messages': [],
    }

    # TODO: app.invoke로 실행하고 단계접 진행 상황 출력


if __name__ == '__main__':
    asyncio.run(main('생성형 AI가 바꾸는 소프트웨어 개발의 미래'))


## 노트북에서 실행

TODO가 남아 있어 정상 실행을 기대하면 안 됩니다. CodeMate와 구현·검토 후 사용하세요. 노트북의 이벤트 루프에서는 원본의 `asyncio.run` 대신 아래처럼 `await`를 사용합니다. API 호출·파일 생성·Langfuse 기록이 발생할 수 있습니다.


In [ ]:
await main('생성형 AI가 바꾸는 소프트웨어 개발의 미래')


## CodeMate에게 요청하고 검토하기

> TODO를 작은 단계로 나누고 첫 단계의 완료 조건과 검증 방법만 제안해 줘. MCP·RAG·승인·파일 부작용도 표시해 줘.

CodeMate가 제안한 수정의 근거를 읽고, 실행 가능한 환경에서 출력·Trace·생성 파일을 확인하세요.


## 확인 질문

1. Writer 전에 승인을 받는가? 배포본에 MCP 서버와 스킬이 있는가? README의 UI 배점과 코드가 맞는가?
2. 실행 전 어떤 환경과 자료를 확인해야 하는가?
3. 결과가 기대와 다르면 어느 단계의 기록부터 볼 것인가?
